# **From Raw Table to Analysis Dataset: Auditing and Preparing 500,000 Calgary Building Permits**

**How an open-data table becomes a dataset that can be trusted:**
- What the table contains?
- what is wrong with it?
- what is decided about each problem?
- what is built from it?

**Author:** Emely Lephilibert.   
**Data:** City of Calgary, Building Permits (dataset c2es-76ed), snapshot of 2026-11-05.   
**Language:** Python   
**Notebook 1 of 2:** this notebook builds the analysis dataset. The second notebook analyses it.

## The problem

Every number in an analysis inherits the problems of the table it comes from. A raw table can hold placeholders in place of missing values, impossible values, dates in the wrong order, and fields that mean different things in different rows. If those problems are not found and decided one by one, they end up inside averages and charts, where nobody can see them.

This notebook starts from the raw table of building permit applications that the City of Calgary publishes, and ends with a dataset that an analysis can use. It shows what the table contains, which problems it has, what is decided about each one and why, and which variables are built on top of it. **The same steps apply to any operational table that arrives from a system nobody designed for analysis, such as claims, loans or service tickets.**

## What this notebook produces

- A **dated snapshot** of the source table, saved exactly as received.
- An **issues log** with every problem found in the audit and the decision taken for each one.
- A **processed dataset** with the cleaned columns, the flags that mark each decision, and the derived variables, with a data dictionary.

Two rules guide the work. No raw value is overwritten: where a value is corrected or set aside, the result goes into a new column or a flag. And every decision is written as code, so the processed dataset can be rebuilt from the snapshot at any time.

## The five chapters

| Chapter | Question | Why it matters |
|---------|----------|----------------|
| 1. The source | Where does the table come from, and is the copy complete? | An analysis can only be repeated if the data are frozen and checked. |
| 2. The audit | What does the table contain, and what is wrong with it? | Problems that are not named are not handled. |
| 3. The decisions | What is done about each problem? | A decision that is written down can be reviewed and reversed. |
| 4. The new variables | Which variables does the analysis need that the table does not have? | Each definition is made once, so every analysis uses the same one. |
| 5. The output | What is saved, and what does each column mean? | The dataset is only useful if it is documented. |

## How to read this notebook

Each chapter opens with its question, shows the evidence in tables and charts, and closes with a **Summary** that gives the question, the answer and the evidence behind it. The City refreshes the table almost every day, so the notebook downloads it once, saves a dated snapshot, and every later step reads that snapshot. The code is included so that every number can be reproduced.

## Setup

The notebook uses pandas for the tables, requests to read the portal and matplotlib for the few charts. The raw snapshot and the processed dataset are saved in two folders, `data/raw` and `data/processed`.

In [1]:
# Import the libraries: web requests, data tables, charts and file handling
import os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from io import StringIO
from pathlib import Path

# Set one chart style for the whole notebook
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

# Define the colors used in all charts
BLUE, ORANGE, GREY = "#2F5D8A", "#D9822B", "#7A7A7A"
LIGHT_BLUE, NAVY = "#9DB7D1", "#1F3B5C"

# Address of the dataset's data endpoint (CSV format)
BASE = "https://data.calgary.ca/resource/c2es-76ed.csv"

# Folders for the snapshot and the processed dataset, relative to the repository root
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Chapter 1. The source

**Question:** Where does the table come from, and is the copy complete?

**Why it matters:** An analysis can only be repeated if the data are frozen and checked. The City refreshes the table almost every day, so a copy that is not dated and verified cannot be compared with another one.

## 1.1 What is kept

The table has columns that the analysis does not need. 21 columns are kept: identifier and status, dates, classification, cost and size, and location. The columns left out are the names of applicants and contractors, the free text and the street addresses (personal information that no question needs), the detail of each location in permits with several locations, and a column that repeats the latitude and the longitude.

In [2]:
# Columns kept in the snapshot, grouped by role
KEEP = [
    # identifier and status
    "permitnum", "statuscurrent",
    # dates
    "applieddate", "issueddate", "completeddate",
    # classification
    "permittype", "permittypemapped", "permitclass", "permitclassgroup", "permitclassmapped",
    "workclass", "workclassgroup", "workclassmapped",
    # size and cost
    "housingunits", "estprojectcost", "totalsqft",
    # location
    "communitycode", "communityname", "latitude", "longitude", "locationcount",
]

# Check that the list has 21 columns and no repeats
print(len(KEEP), len(set(KEEP)))

21 21


## 1.2 The download

The portal total is read before and after the download, and the two must match, so that the copy is a complete picture of the table at one moment. The table is downloaded in pages of 50,000 rows, ordered by `permitnum`, so that no row is skipped or repeated between pages. Every column is read as text and the values "N/A" and "NA" are kept as they are, so nothing is changed at this step.

The copy is saved as a dated file. If the file of that date already exists, the notebook loads it instead of downloading again, so every run works on the same table.

In [3]:
# Date of the snapshot and name of its file (change the date only to download a new snapshot)
SNAP_DATE = "2026-11-05"
SNAP_FILE = RAW_DIR / f"building_permits_{SNAP_DATE}.csv.gz"

# Ask the portal how many rows the table has right now
def portal_total():
    r = requests.get(BASE, params={"$select": "count(*) as n"}, timeout=120)
    r.raise_for_status()                                    # stop with an error if the portal rejects the request
    return int(pd.read_csv(StringIO(r.text))["n"][0])       # read the count from the answer

# Download the kept columns in pages of 50,000 rows and return them as one table of raw text
def download_snapshot(page_size=50_000):
    pages, offset = [], 0
    while True:
        params = {
            "$select": ",".join(KEEP),   # only the 21 kept columns
            "$order": "permitnum",       # fixed order, so pages do not overlap
            "$limit": page_size,         # rows per page
            "$offset": offset,           # where this page starts
        }
        r = requests.get(BASE, params=params, timeout=300)  # ask the portal for one page
        r.raise_for_status()                                # stop with an error if the portal rejects it
        page = pd.read_csv(StringIO(r.text), dtype=str, keep_default_na=False)  # raw text, "N/A" kept
        if page.empty:                                      # no more rows, stop
            break
        pages.append(page)
        offset += page_size
        print(f"Downloaded {offset:,} rows requested")     # simple progress message
    return pd.concat(pages, ignore_index=True)              # join all pages into one table

In [4]:
# Load the stored snapshot if it exists; otherwise download it and check the portal total
if SNAP_FILE.exists():
    snap = pd.read_csv(SNAP_FILE, dtype=str, keep_default_na=False)
    print(f"Loaded the stored snapshot: {SNAP_FILE}")
else:
    total_before = portal_total()
    snap = download_snapshot()
    total_after = portal_total()
    print(f"Portal total before: {total_before:,}")
    print(f"Portal total after:  {total_after:,}")
    print("Table unchanged during the download:", total_before == total_after)

    # Save only a complete copy: the table did not change and every row was received
    if total_before != total_after or len(snap) != total_after:
        raise RuntimeError("The copy is not complete. Run the cell again.")
    snap.to_csv(SNAP_FILE, index=False)
    print(f"Saved: {SNAP_FILE}")

# Check the copy
print(f"Rows: {len(snap):,} | unique permit numbers: {snap['permitnum'].nunique():,} | columns: {snap.shape[1]}")

Loaded the stored snapshot: data/raw/building_permits_2026-11-05.csv.gz
Rows: 503,411 | unique permit numbers: 503,411 | columns: 21


The snapshot has 503,411 rows and 21 columns. The number of unique permit numbers equals the number of rows, and both equal the portal total, which did not change between the start and the end of the download. No row was skipped or repeated between pages, and the copy is a complete picture of the table on 2026-11-05.

## 1.3 Check of the saved file

The file is read back from disk, as text, and compared with the table in memory. The check also confirms that the raw text values "N/A" were kept and not converted to missing values.

In [5]:
# Read the saved file back as raw text, keeping "N/A" as it is
check = pd.read_csv(SNAP_FILE, dtype=str, keep_default_na=False)

# Compare the file with the table in memory
print("Same shape:     ", check.shape == snap.shape)
print("Same content:   ", check.equals(snap))

# Count the raw "N/A" values in the community name column of both tables
print("N/A in memory:  ", (snap["communityname"] == "N/A").sum())
print("N/A in the file:", (check["communityname"] == "N/A").sum())

Same shape:      True
Same content:    True
N/A in memory:   919
N/A in the file: 919


The saved file has the same shape and the same content as the table in memory. The 919 raw "N/A" values in `communityname` were kept, so the snapshot is stored exactly as received from the portal.

## Summary of Chapter 1: The source

**Question: Where does the table come from, and is the copy complete?**

**Answer:** The table is the Building Permits dataset that the City of Calgary publishes on its open data portal. The copy of 2026-11-05 is complete: 503,411 rows, one for each permit number, with 21 columns, saved exactly as received.

**What the evidence shows:**

- The portal total was 503,411 rows before and after the download, and the copy has the same number of rows and of unique permit numbers, so no row was skipped or repeated.
- 21 columns are kept. The names of applicants and contractors, the free text and the street addresses are left out because no question needs them.
- The file read back from disk is identical to the table in memory, and the 919 raw "N/A" values of the community name were kept, so nothing was changed at this step.
- The snapshot is dated and saved. The City refreshes the table almost every day, so every later step reads this file and not the portal, and every run works on the same table.

# Chapter 2. The audit

**Question:** What does the table contain, and what is wrong with it?

**Why it matters:** Problems that are not named are not handled. The audit changes nothing in the data: it counts, lists and describes, and it ends with a log of every problem found, which Chapter 3 uses to decide what to do with each one.

## 2.1 Structure

The snapshot is all text. A working table is made from it and the snapshot stays untouched. Before converting the numeric and date columns, the values that cannot be read as a number or a date are listed, which shows how missing values are written in each column. Then the columns are converted, and the permit number and the window covered by the dates are checked.

In [6]:
# Working copy of the snapshot, the original table stays untouched
df = snap.copy()

# Columns to convert
DATE_COLS = ["applieddate", "issueddate", "completeddate"]
NUM_COLS = ["housingunits", "estprojectcost", "totalsqft", "locationcount", "latitude", "longitude"]

# For each column, list the text values that cannot be read as a number or a date
for col in NUM_COLS + DATE_COLS:
    values = df[col]
    parsed = pd.to_numeric(values, errors="coerce") if col in NUM_COLS else pd.to_datetime(values, errors="coerce")
    bad = values[parsed.isna()].value_counts()          # values that failed to convert, with their counts
    print(col, "->", bad.to_dict() if len(bad) else "all values convert")

housingunits -> all values convert
estprojectcost -> {'': 39092}
totalsqft -> {'': 346705}
locationcount -> {'': 919}
latitude -> all values convert
longitude -> all values convert
applieddate -> all values convert
issueddate -> {'': 18686}
completeddate -> {'': 19722}


In [7]:
# Convert the numeric columns: empty text becomes a missing value, any other bad value raises an error
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col].replace("", pd.NA))

# Convert the date columns in the same way
for col in DATE_COLS:
    df[col] = pd.to_datetime(df[col].replace("", pd.NA))

# Show the type and the number of missing values of each converted column
pd.DataFrame({"type": df[NUM_COLS + DATE_COLS].dtypes.astype(str),
              "missing": df[NUM_COLS + DATE_COLS].isna().sum()})

,type,missing
housingunits,int64,0
estprojectcost,float64,39092
totalsqft,float64,346705
locationcount,float64,919
latitude,float64,0
longitude,float64,0
applieddate,datetime64[ns],0
issueddate,datetime64[ns],18686
completeddate,datetime64[ns],19722


In [8]:
# Check that each row is one application: rows against unique and missing permit numbers
print(f"Rows: {len(df):,} | unique permit numbers: {df['permitnum'].nunique():,} "
      f"| empty permit numbers: {(df['permitnum'] == '').sum()}")

# First and last value of each date column, and whether every date is at midnight
coverage = pd.DataFrame({"first": df[DATE_COLS].min(), "last": df[DATE_COLS].max(),
                         "missing": df[DATE_COLS].isna().sum()})
print("All dates at midnight:", all((df[c].dropna() == df[c].dropna().dt.normalize()).all() for c in DATE_COLS))
coverage

Rows: 503,411 | unique permit numbers: 503,411 | empty permit numbers: 0
All dates at midnight: True


,first,last,missing
applieddate,1999-06-22,2026-11-05,0
issueddate,1999-06-23,2026-11-05,18686
completeddate,1999-06-30,2026-11-04,19722


The only values that cannot be converted are empty text values, so no other text hides in the numeric or date columns and the conversion is safe. Missing values are written as empty text. After the conversion the missing counts are 39,092 for the estimated cost (7.8%), 346,705 for the square footage (68.9%), 18,686 for the issue date (3.7%), 19,722 for the completion date (3.9%) and 919 for the location count (0.2%). The latitude, the longitude and the housing units have no missing values, which does not mean they are complete: placeholders such as zero are checked in the sections that follow.

Each row is one application: there are 503,411 rows, 503,411 unique permit numbers and no empty permit number, so a count of rows is a count of applications. The application dates run from 1999-06-22 to 2026-11-05, the issue dates from 1999-06-23 to 2026-11-05 and the completion dates from 1999-06-30 to 2026-11-04. Every date is stored at midnight, so the time between two dates is measured in whole days. 1999 and 2026 are partial years, and the applications that are still open are observed only up to the date of the snapshot.

## 2.2 Status

Every application has a current status. Counting the rows per status shows how many applications finished, how many ended without a permit and how many are still in progress. The City does not publish a definition for each status, so the reading of each one is an inference from the data. This count also decides how the time to issue is defined, because it can only be measured for applications that were issued.

In [9]:
# Count the applications in each current status, from most to least frequent
status_counts = df["statuscurrent"].value_counts()
status_table = pd.DataFrame({"applications": status_counts,
                             "share (%)": status_counts / len(df) * 100})

# Show the number of distinct statuses and the table
print(f"Distinct statuses: {len(status_table)}")
status_table.style.format({"applications": "{:,.0f}", "share (%)": "{:.2f}"})

Distinct statuses: 24


,applications,share (%)
statuscurrent,,
Completed,"455,782",90.54
Cancelled,"14,902",2.96
Issued Permit,"11,027",2.19
Expired,"7,649",1.52
File Closed,"4,463",0.89
Pre Backfill Phase,"2,445",0.49
Hold,"1,868",0.37
Pending Plans Review Assignment,"1,040",0.21
Refused,874,0.17


There are 24 distinct statuses, but five of them hold 98.1% of the applications: Completed (455,782; 90.5%), Cancelled (14,902; 3.0%), Issued Permit (11,027; 2.2%), Expired (7,649; 1.5%) and File Closed (4,463; 0.9%). The other 19 statuses are each below 0.5%. Several of them describe applications that are still in progress, such as New, In Review, Application Received and the Pending and Hold statuses.

Some applications ended without a permit (Cancelled and Refused) and others are still in progress, so a time to issue can only be measured for applications that were issued. How the statuses are grouped is decided in Chapter 4.

## 2.3 Missing values

A missing value is not always an error. It can mean "not applicable", "has not happened yet" or "not recorded". The first table counts the empty values in every column, and the text placeholders "N/A" and "NA", which are not empty but carry no information. The second table shows how the missing dates depend on the status of the application.

In [10]:
# Count the empty values in each column: empty text in the text columns, missing values in the converted ones
text_cols = [c for c in df.columns if c not in NUM_COLS + DATE_COLS]
empty = pd.concat([(df[text_cols] == "").sum(), df[NUM_COLS + DATE_COLS].isna().sum()])

# Count the text placeholders "N/A" and "NA" in the text columns
placeholders = df[text_cols].isin(["N/A", "NA"]).sum().reindex(empty.index).fillna(0).astype(int)

# Table of the columns that have empty values or placeholders, from the largest gap to the smallest
missing_table = pd.DataFrame({"Empty": empty, "Share (%)": empty / len(df) * 100, "N/A text": placeholders})
missing_table = missing_table[(missing_table["Empty"] > 0) | (missing_table["N/A text"] > 0)]
missing_table.sort_values("Empty", ascending=False).style.format(
    {"Empty": "{:,.0f}", "Share (%)": "{:.1f}", "N/A text": "{:,.0f}"})

,Empty,Share (%),N/A text
totalsqft,"346,705",68.9,0
estprojectcost,"39,092",7.8,0
completeddate,"19,722",3.9,0
issueddate,"18,686",3.7,0
workclassmapped,"4,770",0.9,0
locationcount,919,0.2,0
permitclass,50,0.0,0
communityname,0,0.0,919
communitycode,0,0.0,919


Three columns have large gaps: the square footage (346,705 applications; 68.9%), the estimated cost (39,092; 7.8%) and the dates of completion and issue (19,722 and 18,686; 3.9% and 3.7%). The type of work (`workclassmapped`) is empty in 4,770 applications (0.9%), the location count in 919 (0.2%) and the permit class in 50. The community name and the community code hold the placeholder "N/A" in 919 applications each, instead of an empty value.

In [11]:
# Share of each status with no issue date and with no completion date
dates_by_status = df.groupby("statuscurrent").agg(
    applications=("permitnum", "size"),
    no_issue_date=("issueddate", lambda s: s.isna().mean() * 100),
    no_completion_date=("completeddate", lambda s: s.isna().mean() * 100),
).sort_values("applications", ascending=False)

# Show the table, in percentages
dates_by_status.style.format({"applications": "{:,.0f}", "no_issue_date": "{:.1f}",
                              "no_completion_date": "{:.1f}"})

,applications,no_issue_date,no_completion_date
statuscurrent,,,
Completed,"455,782",0.0,0.0
Cancelled,"14,902",82.6,0.1
Issued Permit,"11,027",0.0,99.8
Expired,"7,649",1.1,0.0
File Closed,"4,463",0.3,0.0
Pre Backfill Phase,"2,445",0.0,100.0
Hold,"1,868",94.1,99.4
Pending Plans Review Assignment,"1,040",100.0,100.0
Refused,874,99.9,0.0


Most missing dates follow the status. Almost every Completed application has both dates. Those that ended without a permit have no issue date: 82.6% of Cancelled and 99.9% of Refused, and the other 17.4% of Cancelled do have one. Applications that were issued and are not finished have an issue date and no completion date: 99.8% of Issued Permit and all of Pre Backfill Phase, Pre Board Phase and Final Phase. The statuses of work still in progress (New, In Review, Application Received and the Pending and Hold statuses) mostly have neither date, with at least 91.6% without an issue date and at least 97.5% without a completion date. The gaps are therefore mostly structural: a date is missing because the step has not happened yet. A few statuses break the pattern, for example 1.1% of Expired and 0.3% of File Closed have no issue date, which section 2.8 counts.

## 2.4 Estimated project cost

The cost is declared by the applicant, so it is an estimate and not a final cost. Two kinds of values need a look: values that cannot be a real cost (negative, zero or very small amounts) and extremely large values. The first table counts the costs by size. The second lists the ten largest costs, with the type, community and status of each application, to see whether they look plausible. The portal does not state the currency of the amounts.

In [12]:
# Costs that are not missing
costs = df["estprojectcost"].dropna()

# Count the costs by size, from impossible values to ordinary ones
sizes = {
    "Negative": costs < 0,
    "Zero": costs == 0,
    "Between 0 and 10": (costs > 0) & (costs < 10),
    "From 10 to below 100": (costs >= 10) & (costs < 100),
    "100 or more": costs >= 100,
}
cost_sizes = pd.DataFrame({"applications": {k: int(v.sum()) for k, v in sizes.items()}})
cost_sizes["share (%)"] = cost_sizes["applications"] / len(costs) * 100

# Show how many costs are exactly 1.00 and the table
print(f"Costs with a value: {len(costs):,} | exactly 1.00: {(costs == 1).sum():,}")
cost_sizes.style.format({"applications": "{:,.0f}", "share (%)": "{:.2f}"})

Costs with a value: 464,319 | exactly 1.00: 497


,applications,share (%)
Negative,8,0.00
Zero,0,0.00
Between 0 and 10,510,0.11
From 10 to below 100,121,0.03
100 or more,"463,680",99.86


Of the 464,319 applications with a cost, 8 have a negative cost and none has a cost of zero. Another 510 (0.11%) have a cost between 0 and 10 dollars, and 497 of them are exactly 1.00, a value that looks like a placeholder and not like a real estimate. Together, the 518 negative and very small costs are hard to accept as real. A further 121 costs (0.03%) are between 10 and 100 dollars, which is small but not impossible, and the other 463,680 (99.86%) are 100 dollars or more.

In [13]:
# The ten largest costs, with the application they belong to
largest = df.nlargest(10, "estprojectcost")[["permitnum", "applieddate", "permitclassgroup",
                                             "communityname", "statuscurrent", "estprojectcost"]]
largest.style.format({"applieddate": "{:%Y-%m-%d}", "estprojectcost": "{:,.0f}"})

,permitnum,applieddate,permitclassgroup,communityname,statuscurrent,estprojectcost
309136,BP2017-12385,2017-09-15,Institutional,ST. ANDREWS HEIGHTS,Completed,"868,255,213"
134584,BP2007-07297,2007-05-18,Commercial,DOWNTOWN COMMERCIAL CORE,Completed,"644,689,500"
455279,BP2024-24370,2024-11-20,Commercial,BELTLINE,Pending Prescreen Review,"639,048,958"
192644,BP2011-00001,2011-01-03,Industrial,CALGARY INTERNATIONAL AIRPORT,Completed,"596,614,379"
376276,BP2021-09865,2021-06-04,Commercial,EAST SHEPARD INDUSTRIAL,Completed,"400,000,000"
237216,BP2013-10778,2013-07-25,Commercial,DOWNTOWN COMMERCIAL CORE,Completed,"376,000,000"
165450,BP2009-04369,2009-05-12,Institutional,SETON,Completed,"371,926,500"
142669,BP2007-15415,2007-10-04,Commercial,DOWNTOWN COMMERCIAL CORE,Completed,"320,000,000"
368173,BP2021-01759,2021-02-03,Commercial,BELTLINE,Completed,"316,961,700"
487669,BP2026-06022,2026-04-15,Commercial,RICARDO RANCH,Pending Plans Review Assignment,"315,000,000"


The ten largest costs go from 315,000,000 to 868,255,213 dollars. Seven are commercial, two institutional and one industrial, and eight of the ten are Completed. The other two, from 2024 and 2026, are still at an early review stage, so their amounts are declared estimates that nobody has reviewed yet. Several are round figures (400,000,000, 376,000,000, 320,000,000 and 315,000,000), as expected of an estimate. None of them is impossible by type or location, but the dataset cannot confirm them. They are extreme values to handle with care in the analysis, not errors.

## 2.5 Area and housing units

Two more size variables describe each project: `totalsqft`, the total area in square feet, and `housingunits`, the number of dwellings the permit adds or changes. This section looks at how many values are missing, zero or negative in each, and where the area is missing, because a variable that is empty for most rows can only be used for the subset where it exists.

In [14]:
# Size of totalsqft and housingunits: missing, negative, zero and positive values
def size_profile(s):
    s = s.dropna()
    return pd.Series({
        "Non-missing": len(s),
        "Negative": (s < 0).sum(),
        "Zero": (s == 0).sum(),
        "Positive": (s > 0).sum(),
        "Median (positive)": s[s > 0].median(),
        "Maximum": s.max(),
    })

size_table = pd.DataFrame({
    "totalsqft": size_profile(df["totalsqft"]),
    "housingunits": size_profile(df["housingunits"]),
})
size_table.loc["Missing"] = [df["totalsqft"].isna().sum(), df["housingunits"].isna().sum()]
size_table.loc["Missing (%)"] = (size_table.loc["Missing"] / len(df) * 100).round(1)

print(f"Rows in the table: {len(df):,}\n")
print(size_table.round(1).to_string(float_format=lambda x: f"{x:,.1f}"))

Rows in the table: 503,411

                   totalsqft  housingunits
Non-missing        156,706.0     503,411.0
Negative                 2.0           2.0
Zero                   652.0     313,172.0
Positive           156,052.0     190,237.0
Median (positive)    1,945.0           1.0
Maximum             41,979.2         715.0
Missing            346,705.0           0.0
Missing (%)             68.9           0.0


`totalsqft` is missing in 346,705 rows (68.9%), so it describes only 156,706 applications. Among those, 2 are negative, 652 are zero and 156,052 are positive, with a median of 1,945 square feet and a maximum of 41,979.2. `housingunits` has no missing values, but 313,172 rows (62.2%) are zero and only 190,237 (37.8%) are positive, with a median of 1 and a maximum of 715. Two values are negative in each variable. Because every row has a value, a zero in `housingunits` cannot be told apart from "not recorded" using this table alone: it may mean a project that adds no dwellings (a commercial building, for example) or a residential project where the number was never entered.

In [15]:
# Where is totalsqft missing? Share by permit type (types with at least 1,000 permits)
by_type = (
    df.assign(sqft_missing=df["totalsqft"].isna(), units_missing=df["housingunits"].isna())
      .groupby("permittypemapped")
      .agg(Permits=("permitnum", "size"),
           sqft_missing=("sqft_missing", "mean"),
           units_missing=("units_missing", "mean"))
)
by_type = by_type[by_type["Permits"] >= 1000].sort_values("Permits", ascending=False)
by_type["Area missing (%)"] = (by_type.pop("sqft_missing") * 100).round(1)
by_type["Units missing (%)"] = (by_type.pop("units_missing") * 100).round(1)

print(by_type.to_string(formatters={"Permits": "{:,}".format}))

                 Permits  Area missing (%)  Units missing (%)
permittypemapped                                             
Building         485,706              67.7                0.0
Demolition        17,705             100.0                0.0


The table splits the rows into two permit types, Building (485,706) and Demolition (17,705). Demolition permits never have an area: all 17,705 are missing, which explains why the gap is not random. Among Building permits the area is missing in 67.7% of the rows, about 329,000 applications, so most of the missing area is not caused by demolitions. Housing units are complete in both types. Because the area is absent for most building permits, any analysis that uses it would cover only about a third of the table, and the comparison between permit types is not possible for demolitions.

### Are the zeros in housing units plausible?

A zero is expected for a commercial or industrial permit, but it is suspicious for a residential one. The next table shows the share of zero housing units by permit class group, to see where the zeros concentrate.

In [16]:
# Share of zero housing units by permit class group (groups with at least 1,000 permits)
units_by_class = (
    df.assign(is_zero=df["housingunits"] == 0,
              is_positive=df["housingunits"] > 0)
      .groupby("permitclassgroup", dropna=False)
      .agg(Permits=("permitnum", "size"),
           Zero=("is_zero", "sum"),
           Positive=("is_positive", "sum"),
           zero_share=("is_zero", "mean"))
)
units_by_class = units_by_class[units_by_class["Permits"] >= 1000].sort_values("Permits", ascending=False)
units_by_class["Zero (%)"] = (units_by_class.pop("zero_share") * 100).round(1)

print(units_by_class.to_string(formatters={"Permits": "{:,}".format,
                                           "Zero": "{:,}".format,
                                           "Positive": "{:,}".format}))

                     Permits    Zero Positive  Zero (%)
permitclassgroup                                       
Single Family        273,527 140,977  132,550      51.5
Garage                64,487  64,443       44      99.9
Commercial            62,530  62,454       75      99.9
Two Family            28,449   5,631   22,818      19.8
Secondary Suites      18,565       0   18,565       0.0
Unspecified           13,736   6,679    7,057      48.6
Townhouse             10,907   5,185    5,722      47.5
Institutional          9,715   9,646       69      99.3
Apartment              7,010   4,884    2,125      69.7
Industrial             5,606   5,602        4      99.9
Special Function Ten   3,521   3,520        1     100.0
General                2,495   2,459       36      98.6
Additional Dwelling    1,158       0    1,158       0.0


Zeros follow the type of project. In the non-residential groups they are almost universal: Garage (99.9%), Commercial (99.9%), Institutional (99.3%), Industrial (99.9%), Special Function Ten (100.0%) and General (98.6%), which is expected because these permits do not add dwellings. Secondary Suites and Additional Dwelling have no zeros at all, as their definition requires at least one dwelling. The residential groups are the ones that need care: 51.5% of Single Family permits are zero (140,977 rows), as are 69.7% of Apartment, 47.5% of Townhouse, 19.8% of Two Family and 48.6% of Unspecified. Single Family alone holds 45% of all zero values in the table. A zero in these groups can be a legitimate project that adds no dwelling, such as an alteration, but this table cannot show that. The two negative values sit in Commercial and Apartment, one in each, which is why those two groups add up to one row less than their total.

### Zero housing units in Single Family permits, by type of work

To see whether the zeros in Single Family are alterations or new construction, the next table crosses them with `workclassgroup`, the grouping of the type of work.

In [17]:
# Single Family permits: share of zero housing units by work class group
sf = df[df["permitclassgroup"] == "Single Family"]

sf_work = (
    sf.assign(is_zero=sf["housingunits"] == 0)
      .groupby("workclassgroup", dropna=False)
      .agg(Permits=("permitnum", "size"),
           Zero=("is_zero", "sum"),
           zero_share=("is_zero", "mean"))
      .sort_values("Permits", ascending=False)
)
sf_work["Zero (%)"] = (sf_work.pop("zero_share") * 100).round(1)

print(sf_work.to_string(formatters={"Permits": "{:,}".format, "Zero": "{:,}".format}))

               Permits    Zero  Zero (%)
workclassgroup                          
New            133,710   1,510       1.1
Improvement    126,986 126,636      99.7
Demolition      12,826  12,826     100.0
Unspecified          5       5     100.0


The zeros in Single Family are almost entirely explained by the type of work. Of the 140,977 zero values, 126,636 are Improvement permits (99.7% of that group), 12,826 are Demolition (100.0%) and 5 are Unspecified, so alterations and demolitions do not add dwellings and a zero is the expected value. Only 1,510 zeros (1.1%) belong to New construction, out of 133,710 permits. Those are the only Single Family rows where a zero is hard to explain, because a new house should add at least one dwelling. The reverse is rare: just 350 Improvement permits (0.3%) have one or more units, which is plausible for an alteration that creates a dwelling. This check covers Single Family only, so the zeros in Apartment, Townhouse and Two Family are not explained by this table.

## 2.6 Location

Each application carries a community (`communitycode`, `communityname`), a pair of coordinates (`latitude`, `longitude`) and `locationcount`, the number of locations linked to the permit. This section checks which of them are missing, set to zero, or fall outside the city, because a coordinate that is empty or wrong would place a permit in the wrong place on any map.

In [18]:
# Coordinates: missing, zero, and outside a rough bounding box around Calgary
LAT_MIN, LAT_MAX = 50.8, 51.3
LON_MIN, LON_MAX = -114.4, -113.8

lat, lon = df["latitude"], df["longitude"]
missing = lat.isna() | lon.isna()
zero = (lat == 0) | (lon == 0)
outside = ~missing & ~zero & ~(lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX))
inside = ~missing & ~zero & ~outside

coord_table = pd.DataFrame({
    "Rows": [missing.sum(), zero.sum(), outside.sum(), inside.sum()],
}, index=["Missing", "Zero", "Outside the box", "Inside the box"])
coord_table["Share (%)"] = (coord_table["Rows"] / len(df) * 100).round(2)

print(f"Bounding box: latitude {LAT_MIN} to {LAT_MAX}, longitude {LON_MIN} to {LON_MAX}\n")
print(coord_table.to_string(formatters={"Rows": "{:,}".format}))
print(f"\nTotal: {coord_table['Rows'].sum():,} of {len(df):,}")


Bounding box: latitude 50.8 to 51.3, longitude -114.4 to -113.8

                   Rows  Share (%)
Missing               0       0.00
Zero                919       0.18
Outside the box       0       0.00
Inside the box  502,492      99.82

Total: 503,411 of 503,411


No row has a missing coordinate, and none falls outside the rectangle drawn around Calgary (latitude 50.8 to 51.3, longitude -114.4 to -113.8; a rough box, not the official city boundary). The problem is a different one: 919 rows (0.18%) have a coordinate equal to zero, which is not a location in Calgary but a placeholder value. The other 502,492 rows (99.82%) have coordinates inside the box. Because the zeros are not missing values, a map or a distance calculation that uses them as they are would place these 919 permits far away from the city.

In [19]:
# Community: empty or N/A, distinct values; and locationcount
for c in ["communityname", "communitycode"]:
    s = df[c]
    empty = (s.isna() | (s.astype(str).str.strip() == "")).sum()
    na = (s == "N/A").sum()
    print(f"{c}: {empty:,} empty, {na:,} 'N/A', {s.nunique():,} distinct values")

print(f"\nlocationcount: {df['locationcount'].isna().sum():,} missing")
lc = df["locationcount"].value_counts().sort_index()
print(lc.head(10).to_string(float_format=lambda x: f"{x:,.0f}"))
print(f"... maximum: {df['locationcount'].max():,.0f}")

communityname: 0 empty, 919 'N/A', 317 distinct values
communitycode: 0 empty, 919 'N/A', 316 distinct values

locationcount: 919 missing
locationcount
1.0     200672
2.0     209510
3.0      70852
4.0       7666
5.0       3253
6.0       1860
7.0       1647
8.0       1118
9.0       1123
10.0       713
... maximum: 638


The community fields have no empty values, but 919 rows have the text "N/A" in both `communityname` and `communitycode`. Without the "N/A" value there are 316 distinct community names and 315 distinct codes, one more name than codes, a small mismatch to keep in mind when grouping by community. `locationcount` is missing in 919 rows, the same count as the "N/A" communities. Among the 502,492 rows that have a value, 200,672 (39.9%) are linked to one location and 209,510 (41.7%) to two, so 81.6% have one or two locations. A few have many more, up to a maximum of 638, which is a very large number of locations for a single permit and is worth treating as an extreme value.

In [20]:
# Do the 919 zero coordinates, N/A communities and missing locationcount coincide?
zero_coord = (df["latitude"] == 0) | (df["longitude"] == 0)
na_comm = df["communityname"] == "N/A"
no_count = df["locationcount"].isna()

print(f"Zero coordinate:          {zero_coord.sum():,}")
print(f"N/A community:            {na_comm.sum():,}")
print(f"Missing locationcount:    {no_count.sum():,}")
print(f"In all three groups:      {(zero_coord & na_comm & no_count).sum():,}")

Zero coordinate:          919
N/A community:            919
Missing locationcount:    919
In all three groups:      919


The three groups are the same 919 applications: every row with a zero coordinate also has "N/A" as community and no `locationcount`. These applications have no location information at all, so they can be kept in the analysis of cost, time and type but cannot be placed on a map or assigned to a community.

## 2.7 Classification

Each permit is described by three pairs of variables: the type (`permittype`, `permittypemapped`), the class (`permitclass`, `permitclassgroup`, `permitclassmapped`) and the work class (`workclass`, `workclassgroup`, `workclassmapped`). The analysis groups permits by these labels, so this section counts how many distinct values each variable has and how many rows carry an uninformative label such as "Unspecified", "N/A" or an empty text.

In [21]:
# Classification variables: distinct values, empty rows and uninformative labels
CLASS_COLS = ["permittype", "permittypemapped",
              "permitclass", "permitclassgroup", "permitclassmapped",
              "workclass", "workclassgroup", "workclassmapped"]

rows = []
for c in CLASS_COLS:
    s = df[c]
    rows.append({
        "Variable": c,
        "Distinct": s.nunique(),
        "Empty": (s.isna() | (s.astype(str).str.strip() == "")).sum(),
        "Unspecified": (s == "Unspecified").sum(),
        "N/A": (s == "N/A").sum(),
        "Unknown": (s == "Unknown").sum(),
        "Other": (s == "Other").sum(),
    })
class_table = pd.DataFrame(rows).set_index("Variable")
class_table["Uninformative (%)"] = (
    class_table[["Empty", "Unspecified", "N/A", "Unknown", "Other"]].sum(axis=1) / len(df) * 100
).round(2)

print(class_table.to_string(formatters={c: "{:,}".format for c in
      ["Distinct", "Empty", "Unspecified", "N/A", "Unknown", "Other"]}))

                  Distinct Empty Unspecified N/A Unknown Other  Uninformative (%)
Variable                                                                         
permittype               5     0           0   0       0     0               0.00
permittypemapped         2     0           0   0       0     0               0.00
permitclass            326    50           0   0       0     0               0.01
permitclassgroup        16     0      13,736   0       0     0               2.73
permitclassmapped        3     0       4,357   0       0     0               0.87
workclass               10     0         483   0       0     0               0.10
workclassgroup           4     0       4,770   0       0     0               0.95
workclassmapped          3 4,770           0   0       0     0               0.95


Two variables are complete: `permittype` (5 distinct values) and `permittypemapped` (2). The others carry uninformative labels, written in two different ways. Four use the text "Unspecified": `permitclassgroup` (13,736 rows, 2.73%), `permitclassmapped` (4,357, 0.87%), `workclassgroup` (4,770, 0.95%) and `workclass` (483, 0.10%). Two use an empty text instead: `workclassmapped` (4,770 rows, 0.95%) and `permitclass` (50, 0.01%). Empty text is not a missing value for pandas, so a simple null count would have reported these two variables as complete. `workclassmapped` has the same count as the Unspecified rows of `workclassgroup`, so it is probably the same information written as an empty text. No variable has "N/A", "Unknown" or "Other" labels. The variables also go from detailed to broad (distinct counts include the uninformative labels): `permitclass` has 326 distinct values, too many to compare directly, while `permitclassgroup` has 16 and `permitclassmapped` only 3; `workclass` has 10, `workclassgroup` 4 and `workclassmapped` 3. The broader versions are easier to use but hide information: `permitclassmapped` has fewer Unspecified rows than `permitclassgroup` (4,357 against 13,736), so the choice of variable changes how many permits can be classified.

In [22]:
# What is inside the groups "Unspecified" and "Special Function Ten"?
for grp in ["Unspecified", "Special Function Ten"]:
    sub = df[df["permitclassgroup"] == grp]
    print(f"permitclassgroup = '{grp}': {len(sub):,} rows")
    top = sub["permitclass"].value_counts().head(6).rename("Rows").to_frame()
    print(top.to_string(formatters={"Rows": "{:,}".format}))
    print()

permitclassgroup = 'Unspecified': 13,736 rows
                                Rows
permitclass                         
9999 - Unspecified            11,261
1706 - Rhs Rowhouse            1,644
9999  Online                     572
1799 - Rhs Accessory Building    119
1701 - Rhs Improvements           60
                                  50

permitclassgroup = 'Special Function Ten': 3,521 rows
                                             Rows
permitclass                                      
7103 - Special Function - Class 1           2,739
7102 - Special Function Tent - Recreational   348
7104 - Special Function - Class 2             327
7101 - Special Function Tent - Commercial     107



Inside the group "Unspecified", 11,261 of the 13,736 rows (82.0%) carry the class "9999 - Unspecified" and 572 more are "9999 Online", so most of them have no class information at all. The second largest class is "1706 - Rhs Rowhouse" with 1,644 rows (12.0%), which is a real residential class that the group did not place, so the group label hides information that `permitclass` still has. The last row of the first table has a blank label, which is examined in the next cell. The group "Special Function Ten" (3,521 rows) holds only special function permits: 2,739 are "Class 1", 327 "Class 2" and 455 are tents (348 recreational and 107 commercial). The name suggests tents, but 87% of the group is not a tent, so the label should be read as special function permits in general.

In [23]:
# What is the label that prints as blank in permitclass?
sub = df[df["permitclassgroup"] == "Unspecified"]
top = sub["permitclass"].value_counts().head(6)
print(repr(top.index[5]), top.iloc[5])

'' 50


The label that printed as blank is an empty text (`''`) and it appears in 50 permits. Since `permitclass` has exactly 50 empty values in total, all of them belong to the group "Unspecified", so the 13,736 rows of that group include 50 whose class is not recorded either.

## 2.8 Dates

Each application has up to three dates: when it was applied for (`applieddate`), issued (`issueddate`) and completed (`completeddate`). The analysis of time depends on these three being in a logical order, so this section checks the order of each pair, whether any date is later than the day the snapshot was taken, and how many days pass between steps, looking for durations that are negative or implausibly long.

In [24]:
# Date order: each check uses only the rows where both dates exist
snap_ts = pd.Timestamp(SNAP_DATE)
a, i, c = df["applieddate"], df["issueddate"], df["completeddate"]

def check(label, mask, valid):
    return {"Check": label, "Rows compared": int(valid.sum()),
            "Rows": int(mask.sum()),
            "Share (%)": round(mask.sum() / valid.sum() * 100, 2)}

both_ai = a.notna() & i.notna()
both_ic = i.notna() & c.notna()
both_ac = a.notna() & c.notna()

checks = pd.DataFrame([
    check("Issued before applied",      (i < a) & both_ai, both_ai),
    check("Issued on the same day",     (i == a) & both_ai, both_ai),
    check("Completed before issued",    (c < i) & both_ic, both_ic),
    check("Completed on the same day as issued", (c == i) & both_ic, both_ic),
    check("Completed before applied",   (c < a) & both_ac, both_ac),
    check("Applied after the snapshot", a > snap_ts, a.notna()),
    check("Issued after the snapshot",  i > snap_ts, i.notna()),
    check("Completed after the snapshot", c > snap_ts, c.notna()),
]).set_index("Check")

print(f"Snapshot date: {SNAP_DATE}\n")
print(checks.to_string(formatters={"Rows compared": "{:,}".format, "Rows": "{:,}".format}))

Snapshot date: 2026-11-05

                                    Rows compared    Rows  Share (%)
Check                                                               
Issued before applied                     484,725       1       0.00
Issued on the same day                    484,725 105,304      21.72
Completed before issued                   470,227      31       0.01
Completed on the same day as issued       470,227     318       0.07
Completed before applied                  483,689       1       0.00
Applied after the snapshot                503,411       0       0.00
Issued after the snapshot                 484,725       0       0.00
Completed after the snapshot              483,689       0       0.00


The three dates are almost always in a logical order. Only 1 application was issued before it was applied for (of 484,725 compared), 31 were completed before being issued (0.01% of 470,227) and 1 was completed before being applied for (of 483,689). No date is later than the snapshot date of 2026-11-05. The large group is a different one: 105,304 applications (21.72%) were issued on the same day they were applied for, and another 318 (0.07%) were completed on the day they were issued. Since every date is stored at midnight, a same-day issue cannot tell an instant approval from one that took several hours, but it is a valid zero-day duration and not an error. Only the 33 rows that break the order are real inconsistencies, and they are 33 different applications.

In [25]:
# Days between steps: extreme durations
dur = pd.DataFrame({
    "Applied to issued":  (i - a).dt.days,
    "Issued to completed": (c - i).dt.days,
    "Applied to completed": (c - a).dt.days,
})

summary = pd.DataFrame({
    "Rows": dur.notna().sum(),
    "Minimum": dur.min(),
    "Median": dur.median(),
    "99th percentile": dur.quantile(0.99),
    "Maximum": dur.max(),
    "More than 10 years": (dur > 3650).sum(),
})
print(summary.to_string(float_format=lambda x: f"{x:,.0f}",
                        formatters={"Rows": "{:,}".format, "More than 10 years": "{:,}".format}))

                        Rows  Minimum  Median  99th percentile  Maximum More than 10 years
Applied to issued    484,725       -6      14              204    4,684                  1
Issued to completed  470,227     -630     160            1,680    6,768                 27
Applied to completed 483,689     -351     180            1,729    6,802                 37


The typical durations are reasonable: the median is 14 days from application to issue, 160 days from issue to completion and 180 days from application to completion. The problems are in the tails. The shortest durations are negative (-6 days from application to issue, -630 from issue to completion and -351 from application to completion), which are the rows that break the date order found before. The longest are very long: the 99th percentile is 204 days for application to issue, 1,680 days (4.6 years) for issue to completion and 1,729 days (4.7 years) for application to completion, and the maximum reaches 4,684 days (12.8 years) to issue and 6,802 days (18.6 years) to complete. 37 applications take more than 10 years from application to completion, 27 of them after being issued. A long project can legitimately stay open for years, so these are not necessarily errors, but they are extreme values that would pull a mean duration and are better described with the median and percentiles.

### Statuses that contradict their dates

The status of an application should agree with its dates: a completed application should have both an issue date and a completion date, and one that is still under review should not have been issued yet. The next table counts the rows where it does not. The statuses are grouped by what they mean in the process: before issue (Hold, In Review, New and the Pending statuses), closed after issue (Expired, File Closed and Suspended-Expired) and open (every status except Completed, Cancelled, Expired, File Closed, Refused and Suspended-Expired). The source does not publish definitions of its 24 statuses, so this grouping is our reading of their names.

In [26]:
# Status versus dates: exact counts of statuses that contradict their dates
st, i, c = df["statuscurrent"], df["issueddate"], df["completeddate"]

BEFORE_ISSUE = ["Hold", "Pending Plans Review Assignment", "In Review", "New",
                "Pending Plans Review", "Application Received",
                "Pending Prescreen Review", "Hold - Prescreen Review",
                "Hold - Pending Plans Review", "Hold - Completeness Review",
                "Pending - Building By-Law Review", "Pending Completeness Review"]
CLOSED_AFTER_ISSUE = ["Expired", "File Closed", "Suspended-Expired"]
NOT_COMPLETED = [s for s in st.unique()
                 if s not in ["Completed", "Cancelled", "Expired", "File Closed",
                              "Refused", "Suspended-Expired"]]

rows = [
    ("Completed, no completion date",
     (st == "Completed") & c.isna(), (st == "Completed")),
    ("Completed, no issue date",
     (st == "Completed") & i.isna(), (st == "Completed")),
    ("Expired, File Closed or Suspended-Expired, no issue date",
     st.isin(CLOSED_AFTER_ISSUE) & i.isna(), st.isin(CLOSED_AFTER_ISSUE)),
    ("Before-issue statuses with an issue date",
     st.isin(BEFORE_ISSUE) & i.notna(), st.isin(BEFORE_ISSUE)),
    ("Open statuses with a completion date",
     st.isin(NOT_COMPLETED) & c.notna(), st.isin(NOT_COMPLETED)),
    ("Cancelled after having been issued",
     (st == "Cancelled") & i.notna(), (st == "Cancelled")),
]
status_checks = pd.DataFrame(
    [{"Check": label, "Rows": int(m.sum()), "Out of": int(d.sum()),
      "Share (%)": round(m.sum() / d.sum() * 100, 2)} for label, m, d in rows]
).set_index("Check")

print(status_checks.to_string(formatters={"Rows": "{:,}".format, "Out of": "{:,}".format}))

                                                          Rows  Out of  Share (%)
Check                                                                            
Completed, no completion date                               25 455,782       0.01
Completed, no issue date                                   194 455,782       0.04
Expired, File Closed or Suspended-Expired, no issue date    98  12,117       0.81
Before-issue statuses with an issue date                   183   5,395       3.39
Open statuses with a completion date                        56  19,736       0.28
Cancelled after having been issued                       2,593  14,902      17.40


The contradictions are few. 25 Completed applications (0.01%) have no completion date and 194 (0.04%) have no issue date. 98 applications that are Expired, File Closed or Suspended-Expired (0.81% of 12,117) have no issue date, although these statuses describe permits that were issued first. In the other direction, 183 applications in a before-issue status (3.39% of 5,395) already have an issue date and 56 in an open status (0.28% of 19,736) already have a completion date. Together these five checks affect at most 556 rows (0.11% of the table), fewer if the groups overlap. The largest group, 2,593 Cancelled applications (17.40% of 14,902), is not a contradiction: they were issued and cancelled afterwards, so they have an issue date because the permit existed for a time.

## 2.9 Issues log

The table below gathers every finding of the audit in one place. Each row has the area it belongs to, its type, the number of rows affected and their share of the 503,411 applications; rows that describe a limitation of the source and not a count of records have no number. The types are: **Missing** (no value), **Anomaly** (a value that cannot be right), **Placeholder** (a value that stands for "unknown"), **Extreme value** (possible but far from the rest), **Structure** (a feature of how the table is built) and **Limitation** (something the source does not tell us). Nothing is changed here: Chapter 3 decides what to do with each entry.

In [27]:
# Issues log: every finding of the audit, with its size computed from the data
n = len(df)
cost, sqft, units = df["estprojectcost"], df["totalsqft"], df["housingunits"]
a, i, c = df["applieddate"], df["issueddate"], df["completeddate"]
zero_coord = (df["latitude"] == 0) | (df["longitude"] == 0)
new_sf_zero = ((units == 0) & (df["permitclassgroup"] == "Single Family")
               & (df["workclassgroup"] == "New"))
out_of_order = (i < a) | (c < i) | (c < a)
sc = status_checks["Rows"]

entries = [
    ("Cost",     "Missing estimated cost", "Missing", cost.isna().sum()),
    ("Cost",     "Negative estimated cost", "Anomaly", (cost < 0).sum()),
    ("Cost",     "Estimated cost from 0 to below 10 dollars (most are exactly 1.00)",
                 "Placeholder", ((cost >= 0) & (cost < 10)).sum()),
    ("Cost",     "Ten largest estimated costs (315,000,000 to 868,255,213)",
                 "Extreme value", (cost >= cost.nlargest(10).min()).sum()),
    ("Cost",     "Currency and price level of the costs are not stated", "Limitation", None),
    ("Area",     "Missing total area (all Demolition permits and most Building permits)",
                 "Missing", sqft.isna().sum()),
    ("Area",     "Total area equal to zero or negative", "Anomaly", (sqft <= 0).sum()),
    ("Units",    "Negative housing units", "Anomaly", (units < 0).sum()),
    ("Units",    "Zero housing units in New Single Family permits", "Anomaly", new_sf_zero.sum()),
    ("Units",    "Zero housing units in all other permits (mostly expected)",
                 "Structure", ((units == 0) & ~new_sf_zero).sum()),
    ("Location", "Coordinates equal to zero, community 'N/A' and no locationcount (same rows)",
                 "Placeholder", zero_coord.sum()),
    ("Location", f"locationcount up to {df['locationcount'].max():,.0f} locations for one permit",
                 "Extreme value", None),
    ("Class",    "Permit class group 'Unspecified'", "Placeholder",
                 (df["permitclassgroup"] == "Unspecified").sum()),
    ("Class",    "Work class group 'Unspecified' (workclassmapped empty in the same count)",
                 "Placeholder", (df["workclassgroup"] == "Unspecified").sum()),
    ("Class",    "permitclass empty text", "Missing",
                 (df["permitclass"].astype(str).str.strip() == "").sum()),
    ("Class",    "Group 'Special Function Ten' is not defined (it holds special function permits)",
                 "Limitation", (df["permitclassgroup"] == "Special Function Ten").sum()),
    ("Dates",    "Dates out of order (at least one of the three comparisons fails)",
                 "Anomaly", out_of_order.sum()),
    ("Dates",    "Issued on the same day as applied (zero days to issue)",
                 "Structure", ((i == a) & i.notna()).sum()),
    ("Dates",    "More than 10 years from application to completion",
                 "Extreme value", ((c - a).dt.days > 3650).sum()),
    ("Dates",    f"Partial first and last years ({a.min().date()} to {a.max().date()})",
                 "Structure", None),
    ("Status",   "Missing issue date (depends on the status)", "Structure", i.isna().sum()),
    ("Status",   "Missing completion date (depends on the status)", "Structure", c.isna().sum()),
    ("Status",   "Completed without a completion date", "Anomaly",
                 sc["Completed, no completion date"]),
    ("Status",   "Completed without an issue date", "Anomaly",
                 sc["Completed, no issue date"]),
    ("Status",   "Expired, File Closed or Suspended-Expired without an issue date", "Anomaly",
                 sc["Expired, File Closed or Suspended-Expired, no issue date"]),
    ("Status",   "Before-issue statuses with an issue date", "Anomaly",
                 sc["Before-issue statuses with an issue date"]),
    ("Status",   "Open statuses with a completion date", "Anomaly",
                 sc["Open statuses with a completion date"]),
    ("Status",   "Cancelled after having been issued", "Structure",
                 sc["Cancelled after having been issued"]),
    ("Status",   f"{df['statuscurrent'].nunique()} distinct statuses with no published definitions",
                 "Limitation", None),
]

issues = pd.DataFrame(entries, columns=["Area", "Finding", "Type", "Rows"])
issues.index = range(1, len(issues) + 1)
issues.index.name = "#"
issues["Share (%)"] = issues["Rows"].apply(lambda r: "" if pd.isna(r) else f"{r / n * 100:.2f}")
issues["Rows"] = issues["Rows"].apply(lambda r: "" if pd.isna(r) else f"{int(r):,}")

pd.set_option("display.max_colwidth", None)
print(f"Issues log: {len(issues)} entries, {n:,} applications\n")
print(issues.to_string())

Issues log: 29 entries, 503,411 applications

        Area                                                                          Finding           Type     Rows Share (%)
#                                                                                                                              
1       Cost                                                           Missing estimated cost        Missing   39,092      7.77
2       Cost                                                          Negative estimated cost        Anomaly        8      0.00
3       Cost                Estimated cost from 0 to below 10 dollars (most are exactly 1.00)    Placeholder      510      0.10
4       Cost                         Ten largest estimated costs (315,000,000 to 868,255,213)  Extreme value       10      0.00
5       Cost                             Currency and price level of the costs are not stated     Limitation                   
6       Area            Missing total area (all Demolition

The log has 29 entries: 3 Missing, 10 Anomaly, 4 Placeholder, 3 Extreme value, 6 Structure and 3 Limitation. A share of 0.00 means fewer than 0.005% of the rows. The largest entries are features of how the table is built and not errors: the total area is missing in 346,705 rows (68.87%), housing units are zero in 311,662 other permits (61.91%, mostly expected), 105,304 applications were issued on the day they were applied for (20.92% of the whole table) and the issue and completion dates are missing in 18,686 (3.71%) and 19,722 rows (3.92%), which depends on the status. The one gap that matters for the cost analysis is the missing estimated cost, in 39,092 rows (7.77%). The ten Anomaly entries are small: the largest is 1,510 New Single Family permits with zero housing units (0.30%), and together they affect at most 2,763 rows (0.55%), fewer if they overlap. Placeholders are also small, from 510 estimated costs below 10 dollars (0.10%) to 13,736 permits in the group "Unspecified" (2.73%). The 3 Limitation entries are not counts of records but things the source does not state: the currency of the costs, what "Special Function Ten" means and the definitions of the 24 statuses.

## Summary of Chapter 2: The audit

**Question: What does the table contain, and what is wrong with it?**

**Answer:** The table holds one row per building permit application, with no duplicate permit numbers, and it is in good shape: the records that cannot be right are few. The audit logged 29 findings. Ten are anomalies that affect at most 0.55% of the rows, and the largest findings are features of how the table is built, such as an area that is missing for most permits, housing units that are zero for most rows and dates that exist only at some stages. Nothing was changed: Chapter 3 decides what to do with each finding.

**What the evidence shows:**
- **Structure.** 503,411 rows with unique permit numbers, applied between 1999-06-22 and 2026-11-05, all dates stored at midnight.
- **Status.** 24 statuses with no published definitions, 90.5% of them Completed. At most 556 rows (0.11%) have a status that contradicts their dates.
- **Missing values.** The total area is missing in 68.9% of the rows (every Demolition permit and 67.7% of Building permits) and the estimated cost in 7.8%. The issue and completion dates are missing in 3.7% and 3.9%, depending on the status.
- **Cost.** 8 costs are negative and 510 are below 10 dollars, 497 of them exactly 1.00, which looks like a placeholder. The ten largest go from 315,000,000 to 868,255,213 dollars and are extreme values, not errors.
- **Area and housing units.** Housing units are zero in 62.2% of the rows, and the zeros follow the type of work: in Single Family, 99.7% of Improvement permits are zero, and only 1,510 New permits (1.1%) are hard to explain.
- **Location.** 919 rows (0.18%) have a zero coordinate, "N/A" as community and no `locationcount`: the same rows, with no location information at all.
- **Classification.** "Unspecified" labels cover 2.73% of the permit class groups and 0.95% of the work class groups, and empty text appears in `workclassmapped` (4,770 rows) and `permitclass` (50). The group "Special Function Ten" is 87% special function permits that are not tents.
- **Dates.** Only 33 rows break the order of the dates. 105,304 applications (20.92%) were issued on the day they were applied for, a valid zero-day duration, and 37 take more than 10 years from application to completion.

# Chapter 3: The decisions

**Question:** What do we do with each finding of the audit, and how do we do it without losing the raw data?

**Why it matters:** Every choice made here changes what the analysis can say. Setting a value to missing, keeping it or flagging it leads to different results, so each choice is written down, applied in code and checked, and a reader can disagree with any of them and trace its effect.

Four rules govern all the decisions:
- **No raw value is overwritten.** When a value is not accepted, a new column ending in `_clean` holds the corrected version, and the original column stays untouched.
- **No row is removed.** A row with one problem may be perfectly good in every other variable, so problems are marked, not deleted.
- **Suspicious rows that stay unchanged get a flag.** Flag columns start with `flag_` and let the analysis include or exclude those rows on purpose.
- **Nothing is imputed.** A missing value stays missing.

## 3.1 Decision for each finding

The first column gives the entry numbers of the issues log in section 2.9, and the last one the column that carries the result.

| Log # | Finding | Decision | Result |
|---|---|---|---|
| 1 to 3 | Missing estimated cost; negative costs; costs below 10 dollars | Missing costs stay missing, with no imputation. The 8 negative costs and the 510 costs below 10 dollars are not accepted as real and are set to missing in the clean column | `estprojectcost_clean` |
| 4 | Ten largest costs | Kept as recorded. The analysis uses medians and percentiles, which are not pulled by extreme values | No change |
| 5 | Currency of the costs not stated | Assumed to be Canadian dollars, written as an assumption in the data dictionary and the limitations | No change |
| 6 and 7 | Missing total area; area equal to zero or negative | Missing stays missing. The 654 zero or negative areas are set to missing in the clean column | `totalsqft_clean` |
| 8 | Negative housing units | The 2 negative values are set to missing | `housingunits_clean` |
| 9 | Zero housing units in New Single Family permits | Kept as recorded and flagged, because a new house should add at least one dwelling | `flag_zero_units_new_sf` |
| 10 | Zero housing units in all other permits | Kept: the zeros are mostly expected (alterations, demolitions, non-residential permits) | No change |
| 11 | Coordinates equal to zero, community "N/A", no `locationcount` | Zero coordinates and "N/A" communities are set to missing, so these 919 permits are not placed on a map or assigned to a community | `latitude_clean`, `longitude_clean`, `communityname_clean`, `communitycode_clean` |
| 12 | `locationcount` up to 638 | Kept and documented as an extreme value | No change |
| 13 | Permit class group "Unspecified" | Kept as its own category. The class is not guessed from other columns | No change |
| 14 and 15 | Empty text in `workclassmapped` and `permitclass` | Written as "Unspecified", so every classification variable uses the same label for the same situation | `workclassmapped_clean`, `permitclass_clean` |
| 16 | "Special Function Ten" not defined | Kept and described in the data dictionary | No change |
| 17 | Dates out of order | The dates are kept and the rows flagged. Durations that would be negative are set to missing when the durations are created in Chapter 4 | `flag_dates_out_of_order` |
| 18 and 19 | Issued the same day; more than 10 years to completion | Kept: a zero-day duration is valid and a long project can stay open for years | No change |
| 20 | Partial first and last years | Kept. The analysis by year uses complete years only | No change |
| 21 and 22 | Missing issue and completion dates | Kept as missing. Durations are created only when both dates exist | No change |
| 23 to 27 | Statuses that contradict their dates | Rows kept and flagged | `flag_status_date_conflict` |
| 28 | Cancelled after having been issued | Kept: it is a real sequence, not an error | No change |
| 29 | 24 statuses with no published definitions | The statuses are grouped into six groups in Chapter 4. The grouping is our reading of their names | `status_group` (Chapter 4) |

## 3.2 Applying the decisions

One cell applies all the decisions to a copy of the table called `data`. The raw columns are not touched: each corrected version goes into a new `_clean` column and each flag into a `flag_` column. The cell ends with a count of the rows that each new column changes or flags, to compare with the sizes found in the audit.

In [28]:
# Apply the decisions to a copy of the table: raw columns stay untouched
data = df.copy()

a, i, c = data["applieddate"], data["issueddate"], data["completeddate"]
st = data["statuscurrent"]

# Log 1 to 3: estimated cost. Negative costs and costs below 10 dollars become missing
data["estprojectcost_clean"] = data["estprojectcost"].mask(data["estprojectcost"] < 10)

# Log 6 and 7: total area equal to zero or negative becomes missing
data["totalsqft_clean"] = data["totalsqft"].mask(data["totalsqft"] <= 0)

# Log 8: negative housing units become missing
data["housingunits_clean"] = data["housingunits"].mask(data["housingunits"] < 0)

# Log 9: zero housing units in New Single Family permits stay as recorded, flagged
data["flag_zero_units_new_sf"] = ((data["housingunits"] == 0)
                                  & (data["permitclassgroup"] == "Single Family")
                                  & (data["workclassgroup"] == "New"))

# Log 11: zero coordinates and "N/A" communities become missing
zero_coord = (data["latitude"] == 0) | (data["longitude"] == 0)
data["latitude_clean"] = data["latitude"].mask(zero_coord)
data["longitude_clean"] = data["longitude"].mask(zero_coord)
data["communityname_clean"] = data["communityname"].mask(data["communityname"] == "N/A")
data["communitycode_clean"] = data["communitycode"].mask(data["communitycode"] == "N/A")

# Log 14 and 15: empty text is written as "Unspecified"
for col in ["permitclass", "workclassmapped"]:
    data[f"{col}_clean"] = data[col].where(data[col].astype(str).str.strip() != "", "Unspecified")

# Log 17: dates out of order, flagged
data["flag_dates_out_of_order"] = (i < a) | (c < i) | (c < a)

# Log 23 to 27: statuses that contradict their dates, flagged
# (BEFORE_ISSUE, CLOSED_AFTER_ISSUE and NOT_COMPLETED were defined in section 2.8)
data["flag_status_date_conflict"] = (
    ((st == "Completed") & (c.isna() | i.isna()))
    | (st.isin(CLOSED_AFTER_ISSUE) & i.isna())
    | (st.isin(BEFORE_ISSUE) & i.notna())
    | (st.isin(NOT_COMPLETED) & c.notna())
)

# Rows changed or flagged by each new column
def n_changed(raw, new):
    return int((~((raw == new) | (raw.isna() & new.isna()))).sum())

applied = pd.Series({
    "estprojectcost_clean": n_changed(data["estprojectcost"], data["estprojectcost_clean"]),
    "totalsqft_clean": n_changed(data["totalsqft"], data["totalsqft_clean"]),
    "housingunits_clean": n_changed(data["housingunits"], data["housingunits_clean"]),
    "latitude_clean": n_changed(data["latitude"], data["latitude_clean"]),
    "longitude_clean": n_changed(data["longitude"], data["longitude_clean"]),
    "communityname_clean": n_changed(data["communityname"], data["communityname_clean"]),
    "communitycode_clean": n_changed(data["communitycode"], data["communitycode_clean"]),
    "permitclass_clean": n_changed(data["permitclass"], data["permitclass_clean"]),
    "workclassmapped_clean": n_changed(data["workclassmapped"], data["workclassmapped_clean"]),
    "flag_zero_units_new_sf": int(data["flag_zero_units_new_sf"].sum()),
    "flag_dates_out_of_order": int(data["flag_dates_out_of_order"].sum()),
    "flag_status_date_conflict": int(data["flag_status_date_conflict"].sum()),
}, name="Rows changed or flagged")

print(f"Rows in data: {len(data):,}   Columns: {data.shape[1]}\n")
print(applied.to_frame().to_string(formatters={"Rows changed or flagged": "{:,}".format}))

Rows in data: 503,411   Columns: 33

                          Rows changed or flagged
estprojectcost_clean                          518
totalsqft_clean                               654
housingunits_clean                              2
latitude_clean                                919
longitude_clean                               919
communityname_clean                           919
communitycode_clean                           919
permitclass_clean                              50
workclassmapped_clean                       4,770
flag_zero_units_new_sf                      1,510
flag_dates_out_of_order                        33
flag_status_date_conflict                     520


Every new column changes exactly the number of rows found in the audit: 518 estimated costs (the 8 negative and the 510 below 10 dollars), 654 areas (652 zero and 2 negative), 2 housing units, 919 values in each of the four location columns, 50 empty permit classes and 4,770 empty work classes. The flags mark 1,510 New Single Family permits with zero housing units, 33 rows with dates out of order and 520 rows with a status that contradicts its dates. The status flag has 520 rows and not the 556 that the five checks of section 2.8 add up to, because some rows fail more than one check and are counted once here. The table now has 33 columns, the 21 original ones plus 9 clean columns and 3 flags, and still 503,411 rows.

## 3.3 Check

Three things must hold after the decisions: no row was added or removed, no raw column changed, and every new column has the size expected from the audit. The next cell tests them, and each test prints PASS or FAIL.

In [29]:
# Check: rows, raw columns and sizes of the changes
tests = []

# 1. No row added or removed, and permit numbers stay unique
tests.append(("Same number of rows as the snapshot", len(data) == len(snap)))
tests.append(("Permit numbers still unique", data["permitnum"].is_unique))

# 2. Raw columns identical to the audited table
raw_same = all(data[col].equals(df[col]) for col in df.columns)
tests.append(("All raw columns unchanged", raw_same))

# 3. Sizes of the changes agree with the issues log
cost_bad = (df["estprojectcost"] < 10).sum()
tests.append(("estprojectcost_clean: 8 negative + 510 below 10 = 518",
              applied["estprojectcost_clean"] == cost_bad == 518))
tests.append(("totalsqft_clean: 652 zero + 2 negative = 654",
              applied["totalsqft_clean"] == (df["totalsqft"] <= 0).sum() == 654))
tests.append(("Four location columns change the same 919 rows",
              applied[["latitude_clean", "longitude_clean",
                       "communityname_clean", "communitycode_clean"]].eq(919).all()))
tests.append(("Clean columns have no empty text left",
              not (data["permitclass_clean"].astype(str).str.strip() == "").any()
              and not (data["workclassmapped_clean"].astype(str).str.strip() == "").any()))
tests.append(("No clean value below its limit",
              (data["estprojectcost_clean"].dropna() >= 10).all()
              and (data["totalsqft_clean"].dropna() > 0).all()
              and (data["housingunits_clean"].dropna() >= 0).all()))

for label, ok in tests:
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
print(f"\n{sum(ok for _, ok in tests)} of {len(tests)} tests passed")

PASS  Same number of rows as the snapshot
PASS  Permit numbers still unique
PASS  All raw columns unchanged
PASS  estprojectcost_clean: 8 negative + 510 below 10 = 518
PASS  totalsqft_clean: 652 zero + 2 negative = 654
PASS  Four location columns change the same 919 rows
PASS  Clean columns have no empty text left
PASS  No clean value below its limit

8 of 8 tests passed


All 8 tests pass. The table still has the same rows as the snapshot, with unique permit numbers, every raw column is identical to the table that was audited, and each new column changes the number of rows expected from the issues log. No value below its limit survives in the clean columns, and no empty text is left in the classification columns. The decisions of Chapter 3 are therefore applied as written.

## Summary of Chapter 3: The decisions

**Question: What do we do with each finding of the audit, and how do we do it without losing the raw data?**

**Answer:** Each of the 29 findings has a written decision, applied in one cell to a copy of the table. The result is 12 new columns, 9 corrected `_clean` columns and 3 flags, while every raw column is intact, no row was removed and nothing was imputed. Eight tests confirm that the changes have the sizes found in the audit.

**What the evidence shows:**
- **Set to missing in a clean column.** 518 estimated costs (8 negative and 510 below 10 dollars), 654 areas (zero or negative), 2 negative housing units and the 919 permits with a zero coordinate and "N/A" community, which lose their location.
- **Relabelled.** 50 empty permit classes and 4,770 empty work classes are written as "Unspecified", so every classification variable uses the same label.
- **Kept and flagged.** 1,510 New Single Family permits with zero housing units, 33 rows with dates out of order and 520 rows whose status contradicts their dates.
- **Kept as recorded.** The ten largest costs, the 105,304 same-day issues, the permits that take more than 10 years, the zero housing units that are expected and the "Unspecified" classes. They are real features of the data and the analysis handles them with medians and percentiles.
- **Left for Chapter 4.** The grouping of the 24 statuses and the durations between dates, which need the clean dates and the flags created here.

# Chapter 4: The new variables

**Question:** Which variables does the analysis need that the table does not have, and how are they defined?

**Why it matters:** The analysis of time, of cost and of growth needs a few variables that are not in the source: the stage of each application, the days between steps and the period of the application. Each one is defined once, here, so every later analysis uses the same definition and a reader can check it.

## 4.1 Variables created

| Section | Variable | Definition |
|---|---|---|
| 4.2 | `status_group` | The 24 statuses in six groups, decided from the dates of each status |
| 4.3 | `days_to_issue` | Days from the application date to the issue date |
| 4.3 | `issued_same_day` | True when `days_to_issue` is 0 |
| 4.3 | `days_to_close` | Days from the issue date to the completion date |
| 4.4 | `application_year`, `application_month` | Year and month of the application date |
| 4.4 | `full_year` | True for applications in complete years, which excludes the first and last years of the table |

## 4.2 Status group

The City does not publish definitions for the 24 values of `statuscurrent`, so each status is placed in a group according to what its dates show in the audit (section 2.3), and not according to an official meaning.

| Group | Statuses | What the dates show |
|---|---|---|
| Completed | Completed | Issue and completion dates in almost every row |
| Cancelled or refused | Cancelled, Refused | Ended without a permit: 82.6% of Cancelled and 99.9% of Refused have no issue date |
| Issued, in progress | Issued Permit, Pre Backfill Phase, Pre Board Phase, Final Phase, Re-instated, Work Restarted | An issue date in almost every row and no completion date |
| Closed after issue | Expired, File Closed, Suspended-Expired | An issue date in almost every row and a completion date in every row |
| Open before issue | Pending Plans Review Assignment, In Review, New, Pending Plans Review, Application Received, Pending Prescreen Review, Pending Completeness Review, Pending - Building By-Law Review | No issue date in almost every row |
| On hold | Hold, Hold - Prescreen Review, Hold - Pending Plans Review, Hold - Completeness Review | No issue date in 94% or more of the rows |

Five statuses are not described by the City (Pre Backfill Phase, Pre Board Phase, Final Phase, Re-instated and Work Restarted). All of them have an issue date in almost every row and almost no completion date, so by the data they behave like issued applications still in progress. The original `statuscurrent` is kept, so anyone can check or change the grouping. The next cell creates the column and shows how the dates behave in each group, which is the test of the grouping.

In [30]:
# Statuses in each group (decided from the dates, see the text above)
STATUS_GROUPS = {
    "Completed": ["Completed"],
    "Cancelled or refused": ["Cancelled", "Refused"],
    "Issued, in progress": ["Issued Permit", "Pre Backfill Phase", "Pre Board Phase",
                            "Final Phase", "Re-instated", "Work Restarted"],
    "Closed after issue": ["Expired", "File Closed", "Suspended-Expired"],
    "Open before issue": ["Pending Plans Review Assignment", "In Review", "New",
                          "Pending Plans Review", "Application Received",
                          "Pending Prescreen Review", "Pending Completeness Review",
                          "Pending - Building By-Law Review"],
    "On hold": ["Hold", "Hold - Prescreen Review", "Hold - Pending Plans Review",
                "Hold - Completeness Review"],
}

# Status -> group lookup; each status must appear in exactly one group
status_to_group = {s: g for g, statuses in STATUS_GROUPS.items() for s in statuses}
assert len(status_to_group) == sum(len(v) for v in STATUS_GROUPS.values()), "A status is repeated"

# Every status of the table must be mapped
unmapped = set(data["statuscurrent"].unique()) - set(status_to_group)
assert not unmapped, f"Statuses without a group: {unmapped}"

data["status_group"] = data["statuscurrent"].map(status_to_group)

# Rows in each group, and how often each date is present
by_group = data.groupby("status_group").agg(
    Applications=("permitnum", "size"),
    issue=("issueddate", lambda s: s.notna().mean() * 100),
    completion=("completeddate", lambda s: s.notna().mean() * 100),
).loc[list(STATUS_GROUPS)]
by_group["Share (%)"] = by_group["Applications"] / len(data) * 100
by_group = by_group.rename(columns={"issue": "With issue date (%)",
                                    "completion": "With completion date (%)"})

print(f"Statuses in the mapping: {len(status_to_group)}   Rows without group: {data['status_group'].isna().sum()}\n")
print(by_group[["Applications", "Share (%)", "With issue date (%)", "With completion date (%)"]]
      .to_string(formatters={"Applications": "{:,}".format},
                 float_format=lambda x: f"{x:,.1f}"))

Statuses in the mapping: 24   Rows without group: 0

                     Applications  Share (%)  With issue date (%)  With completion date (%)
status_group                                                                               
Completed                 455,782       90.5                100.0                     100.0
Cancelled or refused       15,776        3.1                 16.4                      99.9
Issued, in progress        14,341        2.8                100.0                       0.1
Closed after issue         12,117        2.4                 99.2                     100.0
Open before issue           3,397        0.7                  2.1                       0.7
On hold                     1,998        0.4                  5.5                       0.7


All 24 statuses are mapped and no row is left without a group. Completed is 90.5% of the table (455,782 applications), and the other five groups hold between 0.4% and 3.1%. The dates behave as each group predicts. Completed and Closed after issue have a completion date in practically every row, and Issued, in progress has an issue date in every row but almost no completion date (0.1%). Cancelled or refused splits as expected: 16.4% of its applications were issued before they ended, while 99.9% have a completion date. Open before issue and On hold hold the rows that should not yet have been issued: 2.1% and 5.5% have an issue date, together the 183 rows flagged in section 3.2. The grouping separates the stages of an application well enough to use as a variable.

## 4.3 Time to issue and time to close

`days_to_issue` is the number of days from the application date to the issue date, and `days_to_close` the number of days from the issue date to the completion date. Both are counted in whole days, because every date is stored at midnight. Following the decisions of Chapter 3, a duration is empty when one of its two dates is missing or when the dates are out of order, so no duration is negative: `days_to_issue` is empty for the application issued before it was applied for, and `days_to_close` for the 31 applications completed before they were issued. `issued_same_day` is True when `days_to_issue` is 0, False when it is greater than 0 and empty when `days_to_issue` is empty. A same-day issue is a valid zero-day duration and is kept.

In [31]:
# Time to issue and time to close, in whole days; empty when a date is missing or out of order
a, i, c = data["applieddate"], data["issueddate"], data["completeddate"]

valid_issue = i.notna() & (i >= a)
data["days_to_issue"] = (i - a).dt.days.where(valid_issue)

data["issued_same_day"] = ((data["days_to_issue"] == 0).astype("boolean")
                           .where(data["days_to_issue"].notna()))

valid_close = i.notna() & c.notna() & (c >= i)
data["days_to_close"] = (c - i).dt.days.where(valid_close)

dur = data[["days_to_issue", "days_to_close"]]
summary = pd.DataFrame({
    "With a value": dur.notna().sum(),
    "Empty": dur.isna().sum(),
    "Minimum": dur.min(),
    "Median": dur.median(),
    "99th percentile": dur.quantile(0.99),
    "Maximum": dur.max(),
})
print(summary.to_string(float_format=lambda x: f"{x:,.0f}",
                        formatters={"With a value": "{:,}".format, "Empty": "{:,}".format}))

print("\nissued_same_day:")
print(data["issued_same_day"].value_counts(dropna=False).to_string(
    float_format=lambda x: f"{x:,.0f}"))

              With a value  Empty  Minimum  Median  99th percentile  Maximum
days_to_issue      484,724 18,687        0      14              204    4,684
days_to_close      470,196 33,215        0     160            1,680    6,768

issued_same_day:
issued_same_day
False    379420
True     105304
<NA>      18687


`days_to_issue` has a value in 484,724 applications and is empty in 18,687 (the 18,686 without an issue date plus the 1 issued before it was applied for). `days_to_close` has a value in 470,196 and is empty in 33,215 (no issue date, no completion date or completed before issue, including the 31 out of order). Neither duration has a negative value any more: the minimum is 0 in both. The typical values are the ones seen in the audit, a median of 14 days to issue and 160 days to close, with the same long tail (99th percentiles of 204 and 1,680 days, maximums of 4,684 and 6,768). 105,304 applications were issued on the same day, 21.72% of those with a time to issue, and 379,420 later.

## 4.4 Period of the application

`application_year` and `application_month` are the year and the month of `applieddate`. The table does not cover whole years at both ends: the first record is from 1999-06-22 and the last from 2026-11-05, so the first and the last year are partial and would look smaller than the others in any count by year. `full_year` is True for the years in between and False for these two, so the analysis by year can use only complete years, as decided in Chapter 3.

In [32]:
# Year and month of the application, and the complete years
a = data["applieddate"]
data["application_year"] = a.dt.year
data["application_month"] = a.dt.month

first_year, last_year = a.min().year, a.max().year
data["full_year"] = data["application_year"].between(first_year + 1, last_year - 1)

print(f"Application years: {first_year} to {last_year}")
print(f"Complete years:    {first_year + 1} to {last_year - 1}\n")

print(data["full_year"].value_counts().rename("Applications")
      .to_frame().to_string(formatters={"Applications": "{:,}".format}))

print("\nPartial years:")
partial = (data.loc[~data["full_year"]]
           .groupby("application_year")
           .agg(Applications=("permitnum", "size"),
                First=("applieddate", "min"),
                Last=("applieddate", "max")))
print(partial.to_string(formatters={"Applications": "{:,}".format,
                                    "First": lambda d: f"{d:%Y-%m-%d}",
                                    "Last": lambda d: f"{d:%Y-%m-%d}"}))

Application years: 1999 to 2026
Complete years:    2000 to 2025

          Applications
full_year             
True           475,411
False           28,000

Partial years:
                 Applications      First       Last
application_year                                   
1999                    6,991 1999-06-22 1999-12-30
2026                   21,009 2026-01-01 2026-11-05


The applications cover 1999 to 2026, and the complete years are 2000 to 2025. 475,411 applications (94.4%) fall in a complete

## 4.5 Check

The new variables must agree with the data they come from. The next cell tests that every status has a group, that no duration is negative, that the same-day indicator matches the durations, that the periods are consistent with the dates and that the raw columns and the number of rows did not change.

In [33]:
# Check: the new variables agree with the data they come from
tests = []

tests.append(("Same number of rows as the snapshot, permit numbers unique",
              len(data) == len(snap) and data["permitnum"].is_unique))
tests.append(("All raw columns unchanged",
              all(data[col].equals(df[col]) for col in df.columns)))
tests.append(("Every row has a status group", data["status_group"].notna().all()))
tests.append(("No negative duration",
              (data["days_to_issue"].dropna() >= 0).all()
              and (data["days_to_close"].dropna() >= 0).all()))
tests.append(("issued_same_day is True exactly when days_to_issue is 0",
              (data["issued_same_day"].dropna() == (data["days_to_issue"].dropna() == 0)).all()))
tests.append(("issued_same_day is empty exactly when days_to_issue is empty",
              data["issued_same_day"].isna().equals(data["days_to_issue"].isna())))
tests.append(("application_year and application_month agree with applieddate",
              (data["application_year"] == data["applieddate"].dt.year).all()
              and data["application_month"].between(1, 12).all()))
tests.append(("full_year is False only in the first and last year",
              set(data.loc[~data["full_year"], "application_year"])
              == {data["application_year"].min(), data["application_year"].max()}))

for label, ok in tests:
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
print(f"\n{sum(ok for _, ok in tests)} of {len(tests)} tests passed")

PASS  Same number of rows as the snapshot, permit numbers unique
PASS  All raw columns unchanged
PASS  Every row has a status group
PASS  No negative duration
PASS  issued_same_day is True exactly when days_to_issue is 0
PASS  issued_same_day is empty exactly when days_to_issue is empty
PASS  application_year and application_month agree with applieddate
PASS  full_year is False only in the first and last year

8 of 8 tests passed


All 8 tests pass. The table keeps its 503,411 rows and its raw columns, every row has a status group, no duration is negative, `issued_same_day` is True exactly when `days_to_issue` is 0 and empty exactly when it is empty, and the periods agree with the application date. `full_year` is False only for 1999 and 2026. The new variables are therefore consistent with the data they come from.

## Summary of Chapter 4: The new variables

**Question: Which variables does the analysis need that the table does not have, and how are they defined?**

**Answer:** Seven new columns: the stage of the application (`status_group`), the days between its steps (`days_to_issue`, `issued_same_day`, `days_to_close`) and its period (`application_year`, `application_month`, `full_year`). Each is defined once, no duration is negative and eight tests confirm that they agree with the data they come from. The table now has 40 columns and the same 503,411 rows.

**What the evidence shows:**
- **Status group.** The 24 statuses fall in six groups with no row left out. Completed is 90.5% of the table, and the dates behave as each group predicts, for example an issue date in 100.0% of the Issued, in progress group and in only 2.1% of Open before issue.
- **Time to issue.** 484,724 applications have a value, with a median of 14 days and a 99th percentile of 204. 105,304 of them (21.72%) were issued on the day they were applied for.
- **Time to close.** 470,196 applications have a value, with a median of 160 days and a 99th percentile of 1,680.
- **Period.** The complete years are 2000 to 2025, which hold 94.4% of the applications. 1999 (from 22 June) and 2026 (to 5 November) are partial and are left out of any comparison by year.

# Chapter 5: The output

**Question:** What does this notebook deliver, and can the result be trusted and reused?

**Why it matters:** The analysis notebook does not repeat any of the work above. It reads one file, so that file has to be complete, in a known order, checked against the source and described well enough for someone who has not read the notebook to use it correctly.

## 5.1 Columns of the processed dataset

The processed dataset keeps the 21 columns of the snapshot and adds the 19 columns created in Chapters 3 and 4, so it has 40 columns and the 503,411 rows of the snapshot. The raw column and its clean version are kept side by side, so any decision can be checked or undone. The columns are placed in groups:

| Group | Columns |
|---|---|
| Identifier and status | `permitnum`, `statuscurrent`, `status_group` |
| Dates | `applieddate`, `issueddate`, `completeddate` |
| Durations and periods | `days_to_issue`, `issued_same_day`, `days_to_close`, `application_year`, `application_month`, `full_year` |
| Classification | `permittype`, `permittypemapped`, `permitclass`, `permitclass_clean`, `permitclassgroup`, `permitclassmapped`, `workclass`, `workclassgroup`, `workclassmapped`, `workclassmapped_clean` |
| Cost, area and housing units | `estprojectcost`, `estprojectcost_clean`, `totalsqft`, `totalsqft_clean`, `housingunits`, `housingunits_clean` |
| Location | `communitycode`, `communitycode_clean`, `communityname`, `communityname_clean`, `latitude`, `latitude_clean`, `longitude`, `longitude_clean`, `locationcount` |
| Flags | `flag_zero_units_new_sf`, `flag_dates_out_of_order`, `flag_status_date_conflict` |

The next cell builds the table in this order and stops with an error if a column is missing, repeated or not listed.

In [34]:
# Columns of the processed dataset, in groups
PROCESSED_COLS = [
    # identifier and status
    "permitnum", "statuscurrent", "status_group",
    # dates
    "applieddate", "issueddate", "completeddate",
    # durations and periods
    "days_to_issue", "issued_same_day", "days_to_close",
    "application_year", "application_month", "full_year",
    # classification
    "permittype", "permittypemapped", "permitclass", "permitclass_clean",
    "permitclassgroup", "permitclassmapped",
    "workclass", "workclassgroup", "workclassmapped", "workclassmapped_clean",
    # cost, area and housing units
    "estprojectcost", "estprojectcost_clean",
    "totalsqft", "totalsqft_clean",
    "housingunits", "housingunits_clean",
    # location
    "communitycode", "communitycode_clean", "communityname", "communityname_clean",
    "latitude", "latitude_clean", "longitude", "longitude_clean", "locationcount",
    # flags
    "flag_zero_units_new_sf", "flag_dates_out_of_order", "flag_status_date_conflict",
]

# No repeats, and exactly the columns of the working table
assert len(PROCESSED_COLS) == len(set(PROCESSED_COLS)) == 40
assert set(PROCESSED_COLS) == set(data.columns), set(data.columns) ^ set(PROCESSED_COLS)

processed = data[PROCESSED_COLS].copy()
print(f"processed: {processed.shape[0]:,} rows and {processed.shape[1]} columns")
print(f"Columns of the snapshot kept: {len(set(PROCESSED_COLS) & set(snap.columns))} of {snap.shape[1]}")

processed: 503,411 rows and 40 columns
Columns of the snapshot kept: 21 of 21


The processed table has the 503,411 rows of the snapshot and 40 columns, and all 21 columns of the snapshot are kept. The other 19 are the clean columns, flags and variables created in Chapters 3 and 4.

## 5.2 Saving the dataset

The table is saved in Parquet, a format that keeps the type of each column (dates stay dates, numbers stay numbers, True/False stays True/False), is compressed and loads much faster than a CSV. The file name carries the snapshot date. To confirm that nothing is lost in the file, the next cell reads it back and compares it with the table in memory.

In [35]:
# Save the processed dataset in Parquet and read it back to compare
OUT_FILE = PROCESSED_DIR / f"building_permits_analysis_{SNAP_DATE}.parquet"
processed.to_parquet(OUT_FILE, index=False, compression="zstd")

reloaded = pd.read_parquet(OUT_FILE)
print(f"File: {OUT_FILE}")
print(f"Size: {OUT_FILE.stat().st_size / 1_000_000:,.1f} MB")
print(f"Rows and columns: {reloaded.shape[0]:,} and {reloaded.shape[1]}")
print("Same columns, same order:", list(reloaded.columns) == list(processed.columns))
print("Same data types:         ", reloaded.dtypes.equals(processed.dtypes))
print("Same content:            ", processed.reset_index(drop=True).equals(reloaded))

File: data/processed/building_permits_analysis_2026-11-05.parquet
Size: 25.2 MB
Rows and columns: 503,411 and 40
Same columns, same order: True
Same data types:          True
Same content:             True


The dataset is saved as `data/processed/building_permits_analysis_2026-11-05.parquet`, a file of 25.2 MB. Read back, it has the same 503,411 rows and 40 columns, in the same order, with the same data types and the same content, so the file holds exactly the table built in this notebook.

## 5.3 Data dictionary

One row per column of the processed dataset. "Empty" counts the missing values; in the `_clean` columns it includes the values set to missing in Chapter 3. A raw column and its clean version are kept side by side.

| Column | Type | Definition | Empty |
|---|---|---|---|
| `permitnum` | text | Permit number, unique: one row per application | 0 |
| `statuscurrent` | text | Current status as recorded by the City (24 values, not defined by the City) | 0 |
| `status_group` | text | The 24 statuses in six groups: Completed, Cancelled or refused, Issued in progress, Closed after issue, Open before issue, On hold | 0 |
| `applieddate` | date | Date the application was submitted | 0 |
| `issueddate` | date | Date the permit was issued | 18,686 |
| `completeddate` | date | Date the application was closed, whatever the outcome | 19,722 |
| `days_to_issue` | number | Whole days from `applieddate` to `issueddate`; empty if a date is missing or out of order | 18,687 |
| `issued_same_day` | True/False | True when `days_to_issue` is 0; empty when `days_to_issue` is empty | 18,687 |
| `days_to_close` | number | Whole days from `issueddate` to `completeddate`; empty if a date is missing or out of order | 33,215 |
| `application_year` | integer | Year of `applieddate` | 0 |
| `application_month` | integer | Month of `applieddate` (1 to 12) | 0 |
| `full_year` | True/False | True for complete years (2000 to 2025); False for 1999 and 2026 | 0 |
| `permittype` | text | Type of permit as recorded (5 values) | 0 |
| `permittypemapped` | text | Broad type: Building or Demolition | 0 |
| `permitclass` | text | Detailed class of the permit, with its code (326 values); 50 rows hold an empty text | 0 |
| `permitclass_clean` | text | `permitclass` with the empty text written as "Unspecified" | 0 |
| `permitclassgroup` | text | Group of the class (16 values, 13,736 of them "Unspecified") | 0 |
| `permitclassmapped` | text | Broad class (3 values, 4,357 of them "Unspecified") | 0 |
| `workclass` | text | Detailed type of work (10 values) | 0 |
| `workclassgroup` | text | Group of the type of work: New, Improvement, Demolition or Unspecified | 0 |
| `workclassmapped` | text | Broad type of work; 4,770 rows hold an empty text | 0 |
| `workclassmapped_clean` | text | `workclassmapped` with the empty text written as "Unspecified" | 0 |
| `estprojectcost` | number | Estimated project cost declared in the application, in dollars (assumed Canadian) | 39,092 |
| `estprojectcost_clean` | number | `estprojectcost` with negative costs and costs below 10 dollars set to missing | 39,610 |
| `totalsqft` | number | Total area in square feet | 346,705 |
| `totalsqft_clean` | number | `totalsqft` with zero and negative areas set to missing | 347,359 |
| `housingunits` | integer | Dwellings added or changed by the project; zero for most rows | 0 |
| `housingunits_clean` | number | `housingunits` with the 2 negative values set to missing | 2 |
| `communitycode` | text | Community code; 919 rows hold "N/A" | 0 |
| `communitycode_clean` | text | `communitycode` with "N/A" set to missing | 919 |
| `communityname` | text | Community name; 919 rows hold "N/A" | 0 |
| `communityname_clean` | text | `communityname` with "N/A" set to missing | 919 |
| `latitude` | number | Latitude of the permit location; 919 rows are zero | 0 |
| `latitude_clean` | number | `latitude` with the zeros set to missing | 919 |
| `longitude` | number | Longitude of the permit location; 919 rows are zero | 0 |
| `longitude_clean` | number | `longitude` with the zeros set to missing | 919 |
| `locationcount` | number | Number of locations linked to the permit (maximum 638) | 919 |
| `flag_zero_units_new_sf` | True/False | True for New Single Family permits with zero housing units (1,510 rows) | 0 |
| `flag_dates_out_of_order` | True/False | True when the dates are out of order (33 rows) | 0 |
| `flag_status_date_conflict` | True/False | True when the status contradicts the dates (520 rows) | 0 |

## 5.4 Conclusions and limitations



### What this notebook delivers

- A dated snapshot of the City of Calgary Building Permits table (2026-11-05): 503,411 applications, one row per permit number, saved in `data/raw`.
- An audit of that table with 29 findings, each with its size, and a decision for every one of them.
- A processed dataset of 40 columns in `data/processed`: the 21 original columns, 9 corrected `_clean` columns, 3 flags and 7 derived variables. No raw value was overwritten, no row was removed and nothing was imputed.
- A data dictionary for the 40 columns and tests that confirm each step.



### What the analysis has to keep in mind

- The data are in good shape: the anomalies affect at most 0.55% of the rows. The large gaps are structural, such as the missing area and the zero housing units, and they define which questions the table can answer.
- Most of the cost analysis rests on `estprojectcost_clean`, which is empty for 39,610 applications (7.9%) and is an estimate declared in the application, not a final cost.
- Durations are in whole days and exist only where both dates exist. A same-day issue cannot show how many hours the approval took.
- Comparisons by year use `full_year`, which leaves out 1999 and 2026.



### Limitations

- **One snapshot.** The City updates the table, so a later download will have more rows. The frozen file in the repository is what every result in the project refers to.
- **Statuses without definitions.** The City does not define its 24 statuses, so `status_group` is our reading of what their dates show, not an official classification.
- **Cost.** It is an estimate declared at application, and the table does not state its currency, which is assumed to be Canadian dollars. Setting costs below 10 dollars to missing is a decision: another threshold would change a few hundred rows.
- **Area.** `totalsqft_clean` has a value in only 156,052 rows (31.0%), and none for demolitions, so any analysis of size covers a third of the table.
- **Housing units.** A zero is expected for most permits, but outside Single Family the audit did not separate a project that adds no dwellings from a value that was not entered.
- **Location.** 919 permits have no location, so maps and community comparisons leave them out.



### Reproducing the results

The notebook downloads the snapshot only if the dated file is not already in `data/raw`, so running it from the first cell with the repository files rebuilds the processed dataset from the same snapshot. The analysis notebook reads only the processed file.

## Summary of Chapter 5: The output

**Question: What does this notebook deliver, and can the result be trusted and reused?**

**Answer:** A processed dataset of 503,411 rows and 40 columns, saved in Parquet with the snapshot date in its name, with a data dictionary and a list of limitations. The file read back is identical to the table built here, and each step of the notebook is backed by tests that pass.

**What the evidence shows:**
- **Columns.** 40 columns in seven groups: the 21 columns of the snapshot and 19 new ones (9 clean columns, 3 flags and 7 derived variables).
- **File.** 25.2 MB in Parquet. Read back, it has the same rows, columns, data types and content as the table in memory.
- **Dictionary.** Every column has its type, its definition and its number of empty values.
- **Limits of the data.** The area exists for 31.0% of the rows, the estimated cost is missing for 7.9% and the 24 statuses have no official definition.

# Closing summary

This notebook turned the raw City of Calgary Building Permits table into a checked analysis dataset.

| | |
|---|---|
| **Source** | City of Calgary Open Data, Building Permits, snapshot of 2026-11-05 |
| **Rows** | 503,411 applications, one per permit number, applied for between 1999-06-22 and 2026-11-05 |
| **Audit** | 29 findings; the 10 anomalies affect at most 0.55% of the rows |
| **Largest gaps** | Total area missing in 68.9% of the rows; estimated cost missing in 7.8% |
| **Set to missing in clean columns** | 518 costs, 654 areas, 2 housing units and the location of 919 permits |
| **Relabelled** | 4,820 empty classification values written as "Unspecified" |
| **Flagged, kept unchanged** | 1,510 New Single Family permits with zero housing units, 33 rows with dates out of order, 520 rows whose status contradicts their dates |
| **New variables** | `status_group`, `days_to_issue`, `issued_same_day`, `days_to_close`, `application_year`, `application_month`, `full_year` |
| **Output** | 40 columns in Parquet, 25.2 MB, with a data dictionary |
| **Checks** | All tests pass (8 in Chapter 3 and 8 in Chapter 4), and the saved file is identical to the table built here |

No raw value was overwritten, no row was removed and nothing was imputed. The analysis of cost, time and growth is in the second notebook of the project, which reads the processed file and does not repeat any of this work.